# Embedding significant results
### Author: Oscar Heath
### Date: Oct 12th 2026

The file significant_results.parquet contains 800 MWAS results for *totiviridiae* which have corrected p-value < 0.05. We now format the results into a structured string, and embed them using the `all-MiniLM-L6-v2` sentence transformer. We then store, for each finding, the top `k` processed restuls with the closest embeddings (`k` controlled by `BLOCKING_PARAM` variable)

In [1]:
import polars as pl
from sentence_transformers import SentenceTransformer

c:\Users\oscar\Documents\totiviridiae_mwas_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BLOCKING_PARAM = 3

In [3]:
df = pl.read_parquet("../../data/results_with_hypotheses.parquet")

In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
sentences = df["response"].to_list()
# normalised so a dot product equals cosine similarity
embeddings = model.encode(
    sentences,
    batch_size=256,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

using device: cuda


Batches: 100%|██████████| 238/238 [00:35<00:00,  6.66it/s]


In [5]:
# the full n x n similarity matrix (~15GB for 60k rows) doesn't fit in GPU memory,
# so compute top-k a chunk of rows at a time
CHUNK_SIZE = 4096

n = embeddings.shape[0]
top_k_chunks = []
for start in range(0, n, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n)
    sims = embeddings[start:end] @ embeddings.T
    # exclude self-similarity so a row's own result never appears in its own top matches
    rows = torch.arange(end - start, device=sims.device)
    sims[rows, rows + start] = float("-inf")
    top_k_chunks.append(sims.topk(BLOCKING_PARAM, dim=1).indices.cpu())

top_k_indices = torch.cat(top_k_chunks)

In [ ]:
processed_results = df["processed_result"].to_list()
hypotheses = df["response"].to_list()
top_k_indices_list = top_k_indices.tolist()
top_k_results = [
    [processed_results[idx] for idx in row_indices]
    for row_indices in top_k_indices_list
]

df = df.with_columns(
    pl.Series("top_similar_results", top_k_results),
    pl.Series("top_similar_indices", top_k_indices_list),
)

In [11]:
df.write_parquet("../../data/hypotheses_processed_results.parquet")

In [ ]:
# generate a df of pairs of processed results and their top similar results
pairs = []
df_indexed = df.with_row_index("index")

for row in df_indexed.iter_rows(named=True):
    for j in row["top_similar_indices"]:
        pairs.append(
            {
                "row_1": row["index"],
                "row_2": j,
                "result_1": row["processed_result"],
                "result_2": processed_results[j],
                "hypothesis_1": row["response"],
                "hypothesis_2": hypotheses[j],
            }
        )
pairs_df = pl.DataFrame(pairs)
pairs_df.write_parquet("../../data/hypotheses_pairs.parquet")